# 🛰️ Kalam — HPE Private Cloud AI Visualizer & Console (Notebook Edition)

A **single-notebook** visualizer for an **HPE Private Cloud AI (PCAI)** deployment: see the
whole stack at once, then drill into any layer. Everything below reads your *live* cluster —
PCAI runs on Kubernetes, so the K8s workloads **are** the AI Essentials services.

| Tab | What it does |
| --- | --- |
| **🗺️ PCAI Stack** | The overview: a live Mermaid map of the entire PCAI deployment — GreenLake control plane → GPU nodes → MLDE/MLDM/MLIS/Lakehouse/Identity/GPU-Operator/Ingress → served model endpoint → managed VMs. Plus an AI health read. |
| **Dashboard** | Docker + Kubernetes health at a glance |
| **Docker** | List / start / stop / restart / remove containers, stream logs, CVE scan + one-click harden |
| **Kubernetes** | Nodes, deployments, services, pods; rollout restart, scale, delete pod, logs |
| **PCAI Assistant** | RAG chatbot for **HPE Private Cloud AI** — Ask & Diagnose modes, grounded with `[[n]]` citations |
| **DevOps Agent** | Cluster-aware LLM agent that proposes approved Docker/K8s actions + Mermaid topology |
| **🖥️ VMs** | **Manual SSH inventory** — see your VMs, live CPU/mem/disk/uptime, and one-click SSH in |

### LLM / model endpoint options (all switchable in the **Config** cell)
1. **Google Gemini** — `gemini-3-flash-preview` via REST
2. **Local LLM** — Ollama / LM Studio (OpenAI-compatible, `http://localhost:11434/v1`)
3. **Custom model endpoint** — any OpenAI-compatible base URL (HPE **MLIS**, vLLM, OpenAI, …)

> With **no** engine configured, the PCAI assistant still works via **lexical search** and
> returns the raw retrieved HPE docs — it never hard-errors.

---
**Run order:** run the cells top-to-bottom once, then use the UI at the bottom. Re-run the
**Config** cell any time you change providers or the VM inventory.

## 1 · Setup

Requires `requests` and `ipywidgets`. `paramiko` is optional (better SSH); without it the
notebook falls back to your system `ssh` binary. Docker/kubectl/ssh must be on your `PATH`
for the respective features.

In [ ]:
# Install dependencies (safe to re-run; comment out if already installed)
import sys, subprocess
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
try:
    import requests  # noqa
except ImportError:
    _pip("requests")
try:
    import ipywidgets  # noqa
except ImportError:
    _pip("ipywidgets")
# paramiko is optional — uncomment to enable the richer SSH backend:
# _pip("paramiko")
print("Setup OK")

## 2 · Configuration

This is the one cell you edit. Pick your `PROVIDER`, fill credentials, and list your VMs.
Re-run this cell after any change.

In [ ]:
# ============================ CONFIG ============================
# --- LLM / model endpoint ---------------------------------------
# PROVIDER options:
#   'gemini' -> Google Gemini (REST)
#   'local'  -> Ollama / LM Studio (OpenAI-compatible on localhost)
#   'custom' -> any OpenAI-compatible model endpoint (HPE MLIS, vLLM, OpenAI, ...)
PROVIDER = 'gemini'

# Google Gemini
GEMINI_API_KEY = ''                       # or leave '' and set env GEMINI_API_KEY
GEMINI_MODEL   = 'gemini-3-flash-preview'

# Local LLM (Ollama / LM Studio) — OpenAI-compatible
LOCAL_URL   = 'http://localhost:11434/v1'
LOCAL_MODEL = 'qwen2.5-coder:7b'

# Custom OpenAI-compatible model endpoint (e.g. HPE MLIS deployment / vLLM / OpenAI)
CUSTOM_URL   = 'https://your-mlis-endpoint.example.com/v1'
CUSTOM_KEY   = ''                         # deployment token / API key (Bearer)
CUSTOM_MODEL = 'my-model'

# --- Embeddings for the PCAI knowledge base ---------------------
# False -> pure lexical retrieval (fast, offline, no API calls). Recommended default.
# True  -> embed with the selected provider's embedding model (better semantic recall).
USE_EMBEDDINGS   = False
GEMINI_EMBED_MODEL   = 'text-embedding-004'
OPENAI_EMBED_MODEL   = 'nomic-embed-text'   # Ollama example; set to your endpoint's embed model

# --- PCAI knowledge base ----------------------------------------
LIVE_CRAWL = False        # True -> also fetch public HPE docs (needs internet). False -> seed only.
CRAWL_MAX_PAGES = 25

# --- VM inventory (manual SSH) ----------------------------------
# Add your VMs here. 'key' = path to a private key file (or None to use password/agent).
# 'password' is optional (only used by the paramiko backend).
VM_INVENTORY = [
    # {'name': 'gpu-node-1', 'host': '10.0.0.11', 'user': 'ubuntu', 'port': 22,
    #  'key': '~/.ssh/id_rsa', 'password': None},
    # {'name': 'lakehouse',  'host': '10.0.0.20', 'user': 'hpe',    'port': 22,
    #  'key': None, 'password': None},
]

import os
os.environ.setdefault('GEMINI_API_KEY', GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY', ''))
print(f"Provider={PROVIDER}  embeddings={USE_EMBEDDINGS}  crawl={LIVE_CRAWL}  VMs={len(VM_INVENTORY)}")

## 3 · LLM layer — Gemini + OpenAI-compatible (local & custom)

One `llm_chat()` that dispatches to the configured provider. `local` and `custom` share the
OpenAI Chat Completions protocol; only the base URL / key / model differ.

In [ ]:
import os, json, requests

def _openai_target():
    '''Return (base_url, api_key, model) for whichever OpenAI-compatible provider is active.'''
    if PROVIDER == 'custom':
        return CUSTOM_URL, CUSTOM_KEY, CUSTOM_MODEL
    return LOCAL_URL, '', LOCAL_MODEL   # local Ollama/LM Studio (no key)

def _openai_chat(messages, temperature=0.2, num_ctx=8192):
    base, key, model = _openai_target()
    url = base.rstrip('/') + '/chat/completions'
    headers = {'Content-Type': 'application/json'}
    if key:
        headers['Authorization'] = f'Bearer {key}'
    body = {'model': model, 'messages': messages, 'temperature': temperature,
            'options': {'num_ctx': num_ctx}}
    r = requests.post(url, headers=headers, json=body, timeout=180)
    if not r.ok:
        raise RuntimeError(f'Endpoint {url} returned HTTP {r.status_code}: {r.text[:300]}')
    data = r.json()
    return data['choices'][0]['message']['content']

def _gemini_chat(prompt, temperature=0.2):
    key = GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY', '')
    if not key:
        raise RuntimeError('No GEMINI_API_KEY configured.')
    url = (f'https://generativelanguage.googleapis.com/v1beta/models/'
           f'{GEMINI_MODEL}:generateContent?key={key}')
    body = {'contents': [{'parts': [{'text': prompt}]}],
            'generationConfig': {'temperature': temperature}}
    r = requests.post(url, json=body, timeout=180)
    if not r.ok:
        raise RuntimeError(f'Gemini returned HTTP {r.status_code}: {r.text[:300]}')
    data = r.json()
    try:
        return data['candidates'][0]['content']['parts'][0]['text']
    except (KeyError, IndexError):
        return ''

def _flatten(system, history, prompt, ai_name='Assistant'):
    lines = [system, '', 'Conversation so far:']
    for h in history:
        who = 'User' if h.get('role') == 'user' else ai_name
        lines.append(f"{who}: {h.get('content','')}")
    lines.append(f'User: {prompt}')
    lines.append(f'{ai_name}:')
    return '\n'.join(lines)

def llm_chat(system, prompt, history=None, ai_name='Assistant', temperature=0.2, num_ctx=8192):
    '''Provider-agnostic chat. Returns (text, ok, reason). Never raises.'''
    history = history or []
    try:
        if PROVIDER == 'gemini':
            txt = _gemini_chat(_flatten(system, history, prompt, ai_name), temperature)
        else:
            msgs = [{'role': 'system', 'content': system}]
            for h in history:
                msgs.append({'role': 'user' if h.get('role') == 'user' else 'assistant',
                             'content': h.get('content', '')})
            msgs.append({'role': 'user', 'content': prompt})
            txt = _openai_chat(msgs, temperature, num_ctx)
        if not txt:
            return '', False, 'The model returned an empty response.'
        return txt, True, ''
    except Exception as e:
        return '', False, str(e)

def llm_available():
    if PROVIDER == 'gemini':
        return bool(GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY'))
    if PROVIDER == 'custom':
        return bool(CUSTOM_URL)
    return True  # local: assume reachable, fall back gracefully otherwise

print('LLM layer ready.')

## 4 · PCAI knowledge base — seed docs, chunking, hybrid retrieval

Ports the RAG core: curated HPE PCAI seed knowledge, paragraph-aware chunking, lexical +
optional-vector hybrid search. Optionally crawls public HPE docs when `LIVE_CRAWL=True`.

In [ ]:
import json, re, math

SEED_KNOWLEDGE = json.loads(r'''[{"title": "HPE Private Cloud AI \u2014 Overview", "url": "https://www.hpe.com/us/en/private-cloud-ai.html", "text": "HPE Private Cloud AI (PCAI) is a turnkey, co-engineered solution from HPE and NVIDIA (part of \"NVIDIA AI Computing by HPE\") that delivers a complete, secure, private AI stack you can stand up in hours instead of months. It bundles compute (NVIDIA GPUs), NVIDIA Spectrum-X Ethernet networking, HPE GreenLake for File Storage, and a curated software stack behind a single control plane.\n\nCore value: a production-ready environment for inference, retrieval-augmented generation (RAG), fine-tuning, and agentic AI, with governed access to enterprise data, built-in security policies, logging, and audit.\n\nManagement: PCAI is operated and monitored through the HPE GreenLake cloud platform, which provides a unified console to deploy, monitor, update, and govern AI workloads. Cloud administrators use the GreenLake control plane; data scientists and developers use the HPE AI Essentials software layer.\n\nSoftware stack combines NVIDIA AI Enterprise (NVAIE) \u2014 including NVIDIA Inference Microservices (NIM) \u2014 with HPE AI Essentials (built on HPE Ezmeral Unified Analytics; a curated set of open-source and HPE tools). An active HPE service agreement/subscription is required."}, {"title": "PCAI Architecture \u2014 The Layers", "url": "https://developer.hpe.com/platform/hpe-private-cloud-ai/home/", "text": "HPE Private Cloud AI is layered:\n\n1. Hardware / Infrastructure: HPE ProLiant Compute servers with NVIDIA GPUs (NVIDIA L40S, H100 NVL, GH200 NVL2, and newer NVIDIA Blackwell options depending on tier), NVIDIA Spectrum-X Ethernet, and HPE storage (HPE GreenLake for File Storage / data fabric). Delivered as a modular, upgradeable rack; network expansion racks scale the platform to 128 GPUs.\n\n2. Kubernetes platform: All workloads run on Kubernetes. The cluster orchestrates AI services, model servers, and data services as pods/deployments. Most PCAI troubleshooting ultimately involves inspecting Kubernetes pods, events, and node GPU capacity. The NVIDIA GPU Operator installs GPU drivers and configures the NVIDIA container runtime across nodes.\n\n3. Data Lakehouse: A federated, unified data layer (EzPresto / Data Lakehouse Gateway) that gives a single view across heterogeneous storage without moving data \u2014 feeding RAG, analytics, and training.\n\n4. HPE AI Essentials: The developer/data-scientist experience \u2014 model development (MLDE), data management (MLDM), inference serving (MLIS), the data lakehouse, an Import Framework, and GenAI/RAG features.\n\n5. NVIDIA AI Enterprise (NVAIE) + NIM: Optimized inference microservices and enterprise AI libraries.\n\n6. HPE GreenLake control plane: Single pane of glass for provisioning, monitoring, updates, entitlements, and governance.\n\nThe architecture is modular so future NVIDIA/HPE/open-source innovations remain compatible."}, {"title": "HPE AI Essentials \u2014 Components & Included Tools", "url": "https://support.hpe.com/hpesc/public/docDisplay?docId=sd00006503en_us", "text": "HPE AI Essentials is the integrated AI/ML software layer of PCAI, built on HPE Ezmeral Unified Analytics. Key components:\n\n- MLDE (Machine Learning Development Environment): all-in-one deep-learning training platform based on Determined AI. Distributed training, hyperparameter search, experiment tracking, GPU scheduling. Docs: hpe-mlde.determined.ai.\n- MLDM (Machine Learning Data Management): data pipelines / data versioning based on Pachyderm. Data lineage, versioned data repos, pipeline-driven processing. MLDM + MLDE can run in a combined cluster.\n- MLIS (Machine Learning Inference Software/Service): scalable model deployment and serving (backed by Kubernetes/KServe). Most \"failed deployment\" errors originate here.\n- Data Lakehouse / EzPresto / Data Lakehouse Gateway + Import Framework: federate and govern enterprise data for analytics, RAG, and training; import third-party AI apps/frameworks.\n- GenAI / Knowledge Base features: build RAG assistants over your own documents.\n\nCurated open-source tools bundled via Ezmeral Unified Analytics (evergreen, enterprise-supported): Apache Spark, Apache Airflow, Apache Superset, Kubeflow, MLflow, Feast (feature store), Presto SQL (EzPresto), and Ray, plus Jupyter notebooks.\n\nAI Essentials versions referenced in docs include 1.5.2, 1.8.x, 1.9.x, 1.10.x, 1.11.x, and 1.12.x (with air-gapped editions). PCAI platform versions include 1.4, 1.7, and 2026.04.0."}, {"title": "MLIS Troubleshooting \u2014 Failed Deployment", "url": "https://docs.ai-solutions.ext.hpe.com/products/mlis/latest/troubleshooting/failed-deployment/", "text": "When an MLIS inference deployment fails to start serving, the most common root causes are:\n\n1) Insufficient disk size for the model. Large models (multi-billion-parameter LLMs) need enough ephemeral/persistent storage to download and load weights. If disk is too small, the inference service fails to start serving. Fix: increase the disk/storage size in the deployment/packaged-model config and redeploy.\n\n2) Requesting GPUs on a cluster/node without available GPUs. If the deployment requests GPU resources but no GPU-equipped node has free GPU capacity, the pod stays Pending / the deployment fails. Fix: verify GPU nodes exist and have free GPUs (kubectl describe node \u2014 check nvidia.com/gpu allocatable vs allocated), lower the GPU request, or check nodeSelectors/taints/tolerations.\n\n3) Insufficient memory request on the packaged model. If memory request/limit is too low, the model process is OOM-killed or won't schedule. Fix: resubmit/redeploy the packaged model with a higher memory request.\n\nGeneral debugging flow for a failed MLIS deployment:\n- kubectl get pods -n <namespace> \u2014 look for Pending, CrashLoopBackOff, ImagePullBackOff, or OOMKilled.\n- kubectl describe pod <pod> \u2014 read Events (scheduling failures, insufficient cpu/memory/gpu, volume issues).\n- kubectl logs <pod> [-c <container>] \u2014 read the model-server error.\n- Confirm the model artifact/registry is reachable and credentials/secrets are valid."}, {"title": "MLIS \u2014 Deployments, Packaged Models, Registries, Tokens, Autoscaling", "url": "https://docs.ai-solutions.ext.hpe.com/products/mlis/latest/deployments/add/", "text": "MLIS serves models via two core objects:\n- Packaged Model: what to serve. Can reference an external source (S3 bucket or huggingface.co); to reach those you must first configure a Registry (with credentials/token) for that source.\n- Deployment: when/how it runs. Specifies which Packaged Model to deploy and the scaling limits (minReplicas / maxReplicas), the scaling metric, and the target value.\n\nCLI (aioli): create a deployment with\n  aioli deployment create <name> --model <packaged-model> --namespace <ns> --authentication-required --auto-scaling-min-replicas N --auto-scaling-max-replicas M --auto-scaling-metric <metric> --auto-scaling-target <value>\n\nAutoscaling:\n- KPA autoscaler (default) supports \"concurrency\" and \"rps\" metrics.\n- HPA autoscaler supports the \"cpu\" metric.\n\nEndpoint security & tokens: if endpoint security is enabled, every request needs a deployment token in the header: \"Authorization: Bearer <YOUR_ACCESS_TOKEN>\". Create/manage deployment tokens in the UI or via CLI (token create). A 401/403 on the endpoint usually means a missing/expired/incorrect deployment token or the wrong namespace.\n\nCanary rollout: MLIS supports canary rollouts to shift a percentage of traffic to a new model version before full promotion.\n\nCommon endpoint issues: model still loading (readiness probe not yet passing \u2014 wait / raise probe timeout), wrong endpoint URL/namespace, missing token, or the packaged model's registry credentials being invalid."}, {"title": "MLDM (Pachyderm) \u2014 Pipeline & Deployment Troubleshooting", "url": "https://docs.ai-solutions.ext.hpe.com/products/mldm/latest/debug/common-issues/", "text": "MLDM data pipelines run as Kubernetes pods. Most common system-level failures:\n- Malformed or missing credentials preventing connection to object storage, the registry, or external services.\n- OOMKilled or other resource-constraint issues (pods can't schedule on available cluster resources).\n- Network issues connecting to pachd, etcd, or other internal/external resources; or failure to find/pull a docker image from the registry.\n\nDiagnostic signs: a job stuck in a state (starting, merging) or a pod in CrashLoopBackoff indicates a system-level failure.\n\nGet deeper logs:\n  pachctl logs --pipeline=<pipeline_name> --raw\n  pachctl logs --master\n  kubectl logs <pod_name>\n  kubectl get events   (or use the MLDM \"View Kubernetes Events\" page)\n\nSpecific fixes:\n- Pods evicted due to disk pressure: nodes' root volume is too small. Each node's root volume must hold the biggest datum you expect to process anywhere in the DAG plus the output files for that datum. Increase node root volume size.\n- OOM: increase the pipeline's memory request/limit, or use a larger node.\n- File uploads failing with connection errors: pachd or worker sidecars OOM-killed while fetching data from object storage \u2014 increase the pipeline spec's cache_size (default 64M)."}, {"title": "MLDE (Determined) \u2014 Experiment & GPU Troubleshooting", "url": "https://hpe-mlde.determined.ai/latest/", "text": "MLDE is HPE's Determined-based training platform. Common training failures and fixes:\n\n- CUDA out of memory (cudaErrorMemoryAllocation): the GPU can't allocate requested memory. Causes: batch too large, memory fragmentation, or other processes holding GPU memory. Fixes: reduce global/per-slot batch size (memory scales ~linearly with batch size); set PYTORCH_CUDA_ALLOC_CONF=max_split_size_mb=512 to reduce fragmentation; inspect with torch.cuda.memory_stats().\n\n- Experiment stuck in \"Queued\"/\"Pending\": not enough free GPU slots in the resource pool. Check the cluster/resource-pool GPU availability and other running experiments; lower slots_per_trial or free resources.\n\n- Multi-node/distributed training failures: environment variables not propagated to all nodes, so workers can't find Python paths, CUDA toolkit, or custom libraries. Ensure the container image and env are consistent across nodes.\n\n- Experiment errored immediately: usually a code/config error in the model definition or a bad checkpoint/dataset path \u2014 read the trial logs in the MLDE UI (or determined CLI: det experiment logs <id>).\n\nMLDE handles GPU scheduling; if trials won't schedule, the root cause is almost always GPU capacity or a resource-pool/priority configuration."}, {"title": "PCAI \u2014 Installation & GreenLake Onboarding", "url": "https://developer.hpe.com/platform/hpe-private-cloud-ai/home/", "text": "Bringing up HPE Private Cloud AI (high level). Prerequisite: an active HPE service agreement, and access to HPE GreenLake (common.cloud.hpe.com) with the PCAI service subscribed.\n\nThe PCAI setup wizard walks through:\n1. Infrastructure configuration: management network, iLO network, and data network info; discovery of control-plane nodes; vCenter, ESXi, and iLO credentials.\n2. Control plane VM setup: control-plane VM networking and credentials.\n3. Worker node discovery: discover management servers via iLO + data network.\n4. Platform configuration: platform ingress IP, and integration with HPE GreenLake for File Storage.\n5. Verification & access: confirm setup completed, then open the GreenLake AI portal (AI Essentials) to deploy AI solutions and grant access to data scientists/developers.\n\nIf onboarding stalls: verify iLO/network reachability and credentials, DNS resolution for the ingress hostname, NTP/time sync, and that the GreenLake device/subscription is properly claimed/entitled. Device onboarding issues usually trace to network/credential/entitlement problems."}, {"title": "PCAI \u2014 Air-Gapped Deployment", "url": "https://docs.ai-solutions.ext.hpe.com/products/mlis/latest/admin/set-up/air-gapped/", "text": "Running PCAI / HPE AI Essentials air-gapped (no internet) requires mirroring everything locally:\n- A local Docker/container registry hosting mirrors of all required images.\n- A local Python package registry hosting all required Python packages (including those MLIS and its dependencies need).\n- An S3-compliant object store for models and dependencies (e.g., MinIO, Ceph, or OpenIO).\n- For Hugging Face models: build a self-contained container that includes the model, then host that container in the local registry.\n\nImportant limitation: NGC/NIM is NOT supported in air-gapped environments, because NIM requires validation against the NVIDIA NGC registry.\n\nAir-gapped AI Essentials editions exist for versions such as 1.7, 1.10, 1.11, and 1.12. Typical air-gapped errors are ImagePullBackOff (image not mirrored), pip/conda failures (package not in the local Python registry), or model-load failures (model not present in the local S3 store) \u2014 the fix is always to add the missing artifact to the corresponding local mirror."}, {"title": "PCAI \u2014 Data Lakehouse, EzPresto & External S3", "url": "https://support.hpe.com/hpesc/public/docDisplay?docId=a00aie19hen_us&page=ManageClusters/connect-object-stores.html", "text": "PCAI's federated Data Lakehouse (EzPresto / Data Lakehouse Gateway) gives a single, unified SQL view across heterogeneous storage so you can train/fine-tune without moving data.\n\nConnecting data:\n- Administrators connect external data sources, including external S3-compliant object stores (via the S3 Proxy Layer). You provide the endpoint, bucket, and access/secret keys.\n- Spark applications can be configured to access data in an external S3 data source through the S3 Proxy Layer.\n- Supported model/artifact stores include external or internal object stores such as MinIO.\n\nThe Import Framework is an open, extensible mechanism to integrate any AI application, framework, or third-party tool into PCAI. Imported components are then managed through PCAI's unified lifecycle (consistent deployment, monitoring, and governance).\n\nCommon data-connection errors: wrong S3 endpoint/region, invalid access/secret keys, TLS/cert issues to the object store, or network/firewall blocking the endpoint. Verify credentials and endpoint reachability first, then check the connector/gateway logs."}, {"title": "PCAI \u2014 Identity, Access & RBAC (Keycloak)", "url": "https://docs.ai-solutions.ext.hpe.com/products/mldm/latest/set-up/authorization/", "text": "HPE AI Essentials uses Keycloak for Identity and Access Management (IAM): SSO, user roles, access controls, authentication tokens, user isolation, and GPU resource governance.\n\n- LDAP/AD federation: Keycloak verifies credentials against your organization's AD/LDAP without storing passwords locally (User Federation). Configure the LDAP connection in the Keycloak realm.\n- SSO + tokens: users authenticate once to Keycloak; applications receive JWTs whose claims describe the identity, attributes, and a \"groups\" claim listing group memberships.\n- RBAC/ABAC: Keycloak manages roles, permissions, and groups for fine-grained access. MLDM additionally supports its own authorization roles plus Kubernetes RBAC.\n\nCommon auth issues: login fails -> check the Keycloak LDAP/AD federation config and that the user is in the right group; API/endpoint 401/403 -> expired or missing token, or the user's group lacks the required role; new users can't see resources -> group-to-role mapping or namespace/project membership not set."}, {"title": "PCAI \u2014 Ingress, TLS Certificates & DNS", "url": "https://support.hpe.com/hpesc/public/docDisplay?docId=sd00007194en_us&page=GUID-65D1AF49-270B-4C1C-A171-868B6CD5AA42.html", "text": "PCAI exposes services through a Kubernetes ingress gateway with TLS. The platform documents an \"Ingress Gateway SSL Certificate\" procedure and AI Essentials \"SSL Certificates\" (update-cert) steps for installing your own CA-signed certificate. Under the hood this typically uses an ingress controller (e.g., Nginx/Istio), cert-manager for certificate lifecycle, and MetalLB or an external load balancer for the ingress IP.\n\nCommon ingress/TLS/DNS problems and fixes:\n- Browser shows a \"fake\"/default certificate: the hostname in the ingress rules doesn't match the certificate CN/SAN. Ensure spec.rules host == spec.tls host == the cert's CN/SAN.\n- Certificate expired or untrusted: replace it via the platform's ingress/SSL-certificate update procedure; make sure clients trust your CA.\n- Service unreachable by hostname: DNS doesn't resolve the ingress hostname to the ingress/LoadBalancer IP \u2014 fix DNS or /etc/hosts; confirm the ingress IP is assigned.\n- 404/502/503 from ingress: backend Service/pod not ready, wrong service name/port in the ingress, or the ingress controller pod is unhealthy (kubectl get pods -n ingress-... ; kubectl describe ingress)."}, {"title": "PCAI \u2014 Common Kubernetes Error Patterns", "url": "https://support.hpe.com/hpesc/public/docDisplay?docId=sd00007592en_us", "text": "Because PCAI runs on Kubernetes, most operational errors surface as pod states. Quick reference:\n\n- Pending: no node can satisfy the pod \u2014 insufficient CPU/memory or no free GPU (nvidia.com/gpu), or an unsatisfiable nodeSelector/taint. Check \"kubectl describe pod\" Events and node allocatable GPUs.\n- ImagePullBackOff / ErrImagePull: image name wrong, private-registry auth missing, or (air-gapped) image not mirrored into the local registry. Verify imagePullSecrets and that the image exists internally.\n- CrashLoopBackOff: container starts then exits. Read \"kubectl logs --previous\". Causes: bad config/env, missing model files, license/entitlement not applied, GPU driver mismatch.\n- OOMKilled (exit 137): memory limit too low for the model \u2014 raise memory request/limit.\n- 0/1 Ready / readiness probe failing: service booted but health check fails \u2014 often still loading a large model (increase probe initialDelay/timeout) or a dependency (data service, vector DB) is down.\n- GPU \"unable to find GPU\" / CUDA errors: NVIDIA GPU Operator / device plugin unhealthy, or driver/toolkit version mismatch. Check the nvidia-device-plugin and gpu-operator pods.\n- Evicted / DiskPressure: node disk full \u2014 clean up or grow the node root volume.\n\nAlways correlate with GreenLake alerts and the AI Essentials UI for the specific service."}, {"title": "PCAI \u2014 Access, Console, and Day-2 Operations", "url": "https://support.hpe.com/hpesc/public/docDisplay?docId=sd00005025en_us", "text": "Accessing PCAI:\n- Cloud administrators manage the system through HPE GreenLake (provisioning, monitoring, software updates, entitlements/licenses, user & access management, health/alerts).\n- Data scientists and app developers work in the HPE AI Essentials web UI (launch notebooks, train with MLDE, manage data with MLDM, deploy models with MLIS, connect data via the lakehouse, and build GenAI/RAG apps).\n\nCommon day-2 tasks:\n- Add/entitle users and assign roles (RBAC) via GreenLake identity + Keycloak.\n- Apply software/firmware updates pushed through GreenLake.\n- Monitor GPU utilization, node health, and workload status.\n- Manage entitlements/licenses \u2014 an unapplied or expired entitlement can cause services to refuse to start.\n\nIf a whole service is down, check in order: (1) GreenLake system health/alerts, (2) Kubernetes node readiness, (3) the specific AI Essentials service pods, (4) entitlement/license status."}, {"title": "PCAI \u2014 Backup, Data Protection & Upgrades", "url": "https://support.hpe.com/connect/s/product?language=en_US&kmpmoid=1014847366&tab=manuals", "text": "Platform lifecycle and data protection for HPE Private Cloud (incl. PCAI):\n- Software/firmware updates are delivered and applied through HPE GreenLake; always follow the version-specific Administration Guide for the upgrade sequence and pre-checks.\n- Data protection integrations: HPE StoreOnce, HPE Zerto (continuous data protection and live workload migration from VMware), and Veeam Data Platform (agentless, host-level, image-based backup with changed-block tracking and cross-platform recovery).\n- HPE Morpheus provides a unified cloud operating model / upgrade path for private cloud management; unified management of VMs and containers on HPE Private Cloud is on the roadmap (GA targeted Q3 2026).\n\nBefore upgrading: check current version and target version compatibility, back up critical data (models, pipeline repos, configs), verify entitlements, and schedule a maintenance window. After upgrading: verify node readiness, GPU operator health, and that each AI Essentials service (MLDE/MLDM/MLIS/lakehouse) comes back healthy."}, {"title": "PCAI \u2014 Sizing Tiers & GPU Options", "url": "https://www.hpe.com/us/en/collaterals/collateral.a50009216enw.html", "text": "HPE Private Cloud AI ships in sizing tiers (described in HPE QuickSpecs, commonly Small / Medium / Large / Extra Large) that differ by GPU compute, CPU, memory, and storage. Smaller tiers target inference/RAG and departmental use; larger tiers target heavy fine-tuning/training and many concurrent workloads.\n\nGPU options vary by tier and generation \u2014 for example NVIDIA L40S on smaller configurations, NVIDIA H100 NVL and GH200 NVL2 on larger ones, with NVIDIA Blackwell-based options on newer configurations. Networking uses NVIDIA Spectrum-X Ethernet; network expansion racks allow scaling toward 128 GPUs.\n\nFor the precise CPU/GPU/RAM/storage bill of materials for a specific order, consult the current HPE PCAI QuickSpecs document (the numbers change across releases, so treat any specific figure as version-dependent and verify against the QuickSpecs for your order)."}]''')
CRAWL_TARGETS = json.loads(r'''["https://developer.hpe.com/platform/hpe-private-cloud-ai/home/", "https://docs.ai-solutions.ext.hpe.com/products/", "https://docs.ai-solutions.ext.hpe.com/products/mlis/latest/", "https://docs.ai-solutions.ext.hpe.com/products/mlis/latest/troubleshooting/failed-deployment/", "https://docs.ai-solutions.ext.hpe.com/products/mldm/latest/debug/common-issues/", "https://hpe-mlde.determined.ai/latest/", "https://docs.pachyderm.com/products/mldm/latest/debug/common-issues/"]''')

STOP = set('the a an and or of to in on for is are be with as at by it this that from you your can will if not but has have was were'.split())

def tokenize(text):
    toks = re.findall(r'[a-z0-9_.-]+', text.lower())
    return [t for t in toks if len(t) > 1 and t not in STOP]

def chunk_text(text, max_len=1200, overlap=150):
    clean = re.sub(r'[ \t]+', ' ', text.replace('\r', '')).strip()
    if len(clean) <= max_len:
        return [clean] if clean else []
    paras = re.split(r'\n\s*\n', clean)
    chunks, buf = [], ''
    for p in paras:
        if len(buf + '\n\n' + p) > max_len and buf:
            chunks.append(buf.strip())
            buf = buf[max(0, len(buf) - overlap):] + '\n\n' + p
        else:
            buf = (buf + '\n\n' + p) if buf else p
        while len(buf) > max_len:
            chunks.append(buf[:max_len].strip())
            buf = buf[max_len - overlap:]
    if buf.strip():
        chunks.append(buf.strip())
    return chunks

def cosine(a, b):
    dot = sum(x*y for x, y in zip(a, b))
    na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

def _strip_html(html):
    html = re.sub(r'(?is)<(script|style|nav|footer|header)[^>]*>.*?</\1>', ' ', html)
    text = re.sub(r'(?s)<[^>]+>', ' ', html)
    text = re.sub(r'&[a-z]+;', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def _crawl(max_pages):
    import requests
    docs = []
    for url in CRAWL_TARGETS[:max_pages]:
        try:
            r = requests.get(url, timeout=15, headers={'User-Agent': 'KalamNB/1.0'})
            if r.ok and r.text:
                title_m = re.search(r'(?is)<title>(.*?)</title>', r.text)
                title = _strip_html(title_m.group(1)) if title_m else url
                body = _strip_html(r.text)
                if len(body) > 200:
                    docs.append({'title': title[:120], 'url': url, 'text': body[:8000]})
        except Exception as e:
            print(f'  crawl skip {url}: {e}')
    return docs

KB = {'chunks': [], 'embedProvider': 'none', 'embedModel': 'lexical', 'updatedAt': None}

def _embed_texts(texts):
    import requests
    if not USE_EMBEDDINGS:
        return None
    try:
        if PROVIDER == 'gemini':
            key = GEMINI_API_KEY or os.environ.get('GEMINI_API_KEY', '')
            if not key: return None
            out = []
            for t in texts:
                u = f'https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_EMBED_MODEL}:embedContent?key={key}'
                r = requests.post(u, json={'content': {'parts': [{'text': t}]}}, timeout=60)
                if not r.ok: return None
                out.append(r.json()['embedding']['values'])
            return out
        base, k, _ = _openai_target()
        u = base.rstrip('/') + '/embeddings'
        h = {'Content-Type': 'application/json'}
        if k: h['Authorization'] = f'Bearer {k}'
        r = requests.post(u, headers=h, json={'model': OPENAI_EMBED_MODEL, 'input': texts}, timeout=120)
        if not r.ok: return None
        return [d['embedding'] for d in r.json()['data']]
    except Exception as e:
        print('  embedding failed, using lexical:', e)
        return None

def train(progress=print):
    docs = list(SEED_KNOWLEDGE)
    progress(f'Seed docs: {len(docs)}')
    if LIVE_CRAWL:
        progress('Crawling public HPE docs...')
        crawled = _crawl(CRAWL_MAX_PAGES)
        progress(f'Crawled {len(crawled)} pages')
        docs += crawled
    chunks = []
    for d in docs:
        for i, ct in enumerate(chunk_text(d['text'])):
            chunks.append({'id': f"{d['url']}#{i}", 'title': d['title'], 'url': d['url'],
                           'text': ct, 'tokens': tokenize(d['title'] + ' ' + ct), 'embedding': None})
    provider, model = 'none', 'lexical'
    if USE_EMBEDDINGS:
        progress('Embedding chunks...')
        vecs = _embed_texts([c['text'] for c in chunks])
        if vecs and len(vecs) == len(chunks):
            for c, v in zip(chunks, vecs): c['embedding'] = v
            provider = PROVIDER
            model = GEMINI_EMBED_MODEL if PROVIDER == 'gemini' else OPENAI_EMBED_MODEL
            progress('Embeddings attached.')
        else:
            progress('Embeddings unavailable -> lexical retrieval.')
    import datetime
    KB.update({'chunks': chunks, 'embedProvider': provider, 'embedModel': model,
               'updatedAt': datetime.datetime.now().isoformat()})
    progress(f'KB ready: {len(chunks)} chunks (provider={provider}).')
    return KB

def search_kb(query, k=8):
    q_tokens = set(tokenize(query))
    q_embed = None
    if KB['embedProvider'] != 'none':
        qv = _embed_texts([query])
        q_embed = qv[0] if qv else None
    use_vec = q_embed is not None and any(c.get('embedding') for c in KB['chunks'])
    scored = []
    for c in KB['chunks']:
        ctok = set(c['tokens'])
        lex = (sum(1 for t in q_tokens if t in ctok) / len(q_tokens)) if q_tokens else 0
        vec = cosine(q_embed, c['embedding']) if (use_vec and c.get('embedding')) else 0
        score = (vec*0.8 + lex*0.2) if use_vec else lex
        if score > 0.001:
            h = dict(c); h['score'] = score; scored.append(h)
    scored.sort(key=lambda x: x['score'], reverse=True)
    # diversify: <=3 chunks per source
    per, kept = {}, []
    for h in scored:
        n = per.get(h['url'], 0)
        if n >= 3: continue
        per[h['url']] = n + 1; kept.append(h)
        if len(kept) >= k: break
    return kept

# Build the KB now with the seed knowledge (train() again to refresh / crawl).
train()
print('PCAI knowledge base initialized.')

## 5 · PCAI Assistant — grounded Ask / Diagnose

Retrieves the top HPE chunks, builds the grounded system instruction, and answers with
inline `[[n]]` citations. If no LLM is reachable it returns the retrieved docs directly
(never a hard error).

In [ ]:
def _pcai_system(mode, context, weak):
    weak_note = ('\nIMPORTANT: The retrieved documentation is a weak match for this query. '
                 'Lead with an honest caveat that your indexed HPE docs may not directly cover this, '
                 'answer with general HPE PCAI / Kubernetes best practice, and recommend the exact HPE '
                 'doc, GreenLake screen, or command to confirm. Do NOT fabricate HPE-specific specifics.\n'
                 if weak else '')
    diag = ('\nThe user has pasted an ERROR, log, or stack trace. Do this:\n'
            '1. State the most likely root cause in one line.\n'
            '2. Give concrete, ordered fix steps (exact kubectl / GreenLake / AI Essentials actions).\n'
            '3. Note what to check to confirm the fix.\n'
            "Be specific to HPE PCAI (Kubernetes-based). If it clearly isn't PCAI-related, say so."
            if mode == 'diagnose' else '')
    return ('You are the HPE Private Cloud AI (PCAI) Assistant inside the Kalam console. You are an '
            'expert on HPE Private Cloud AI, HPE AI Essentials (MLDE, MLDM, MLIS), the data lakehouse, '
            'NVIDIA AI Enterprise/NIM, HPE GreenLake management, and the Kubernetes platform PCAI runs on.\n\n'
            'Answer ONLY using the HPE documentation context below plus well-established Kubernetes/NVIDIA '
            'general knowledge. Ground every specific claim in the context. Cite sources inline using the '
            '[[n]] markers that correspond to the numbered context entries. If the context does not contain '
            "the answer, say clearly what you don't have and suggest which HPE doc or command would resolve "
            'it — do NOT invent HPE-specific details, version numbers, or menu paths.\n'
            f'{weak_note}{diag}\n\n'
            '=== HPE PCAI DOCUMENTATION CONTEXT ===\n'
            f'{context}\n'
            '=== END CONTEXT ===\n\n'
            'Formatting: concise markdown. Headings/bullets for steps. Shell commands in code blocks. '
            'End with a "Sources" list of the [[n]] references you actually used.')

def _doc_fallback(hits, reason):
    body = '\n\n'.join(f"### [{i+1}] {h['title']}\n{h['text']}\n\n_Source: {h['url']}_"
                       for i, h in enumerate(hits))
    return f'> {reason}\n\nHere is the most relevant HPE PCAI documentation I found:\n\n{body}'

def pcai_chat(prompt, mode='ask', history=None):
    '''Returns (answer_markdown, sources_list).'''
    if not KB['chunks']:
        return '**Knowledge base is empty.** Run `train()` first.', []
    hits = search_kb(prompt, k=8)
    context = '\n\n---\n\n'.join(f"[[{i+1}]] Source: {h['title']} ({h['url']})\n{h['text']}"
                                 for i, h in enumerate(hits))
    sources = [{'ref': i+1, 'title': h['title'], 'url': h['url'], 'score': round(h['score'], 3)}
               for i, h in enumerate(hits)]
    top = hits[0]['score'] if hits else 0
    weak = (not hits) or (top < 0.35 if KB['embedProvider'] != 'none' else top < 0.12)
    system = _pcai_system(mode, context, weak)
    if not llm_available():
        return _doc_fallback(hits, 'No AI engine configured — showing retrieved HPE docs directly:'), sources
    text, ok, reason = llm_chat(system, prompt, history or [], ai_name='Assistant')
    if not ok:
        return _doc_fallback(hits, f'{reason} Showing the retrieved HPE docs instead.'), sources
    return text, sources

print('PCAI assistant ready. Try: print(pcai_chat("Why is my MLIS deployment Pending?", "diagnose")[0])')

## 6 · Docker console

Wraps the local `docker` CLI. Includes the vulnerability-scan heuristic + one-click harden
(rebuilds a container on a minimal base image) ported from the server.

In [ ]:
import subprocess, json, re

DOCKER_ID_RE = re.compile(r'^[a-fA-F0-9]{12,64}$|^[a-zA-Z0-9_.-]+$')
NAME_RE = re.compile(r'^[a-zA-Z0-9_.-]+$')

def run_cmd(args):
    '''Run a command (list form). Returns (stdout, stderr, ok).'''
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=120)
        return p.stdout, p.stderr, p.returncode == 0
    except Exception as e:
        return '', str(e), False

def docker_status():
    out, _, ok = run_cmd(['docker', '--version'])
    _, _, running = run_cmd(['docker', 'ps'])
    return {'installed': ok, 'version': out.strip() or 'Not found', 'running': running}

def docker_containers():
    out, err, ok = run_cmd(['docker', 'ps', '-a', '--format', '{{json .}}'])
    if not ok:
        return []
    conts = []
    for line in out.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            p = json.loads(line)
            conts.append({'id': p.get('ID'), 'name': p.get('Names'), 'image': p.get('Image'),
                          'status': p.get('Status'),
                          'state': p.get('State') or ('running' if 'up' in (p.get('Status') or '').lower() else 'exited'),
                          'ports': p.get('Ports'), 'created': p.get('RunningFor') or p.get('CreatedAt')})
        except json.JSONDecodeError:
            pass
    return conts

def docker_action(action, cid):
    if not cid or not DOCKER_ID_RE.match(cid):
        return 'Invalid container id'
    cmd = {'start': ['docker', 'start', cid], 'stop': ['docker', 'stop', cid],
           'restart': ['docker', 'restart', cid], 'remove': ['docker', 'rm', '-f', cid]}.get(action)
    if not cmd:
        return 'Invalid action'
    out, err, ok = run_cmd(cmd)
    return (out.strip() or f'{action} ok') if ok else f'Failed: {err}'

def docker_logs(cid, tail=150):
    if not cid or not DOCKER_ID_RE.match(cid):
        return 'Invalid container id'
    out, err, ok = run_cmd(['docker', 'logs', '--tail', str(tail), cid])
    return out + (f'\n--- STDERR ---\n{err}' if err else '')

def docker_scan(image_name):
    '''Real docker scout quickview if present, else a realistic heuristic (as in the app).'''
    out, err, ok = run_cmd(['docker', 'scout', 'quickview', image_name])
    base = image_name.split(':')[0]
    if ok:
        def _n(word):
            m = re.search(rf'([0-9]+)\s+{word}', out or err, re.I)
            return int(m.group(1)) if m else 0
        summ = {'critical': _n('critical'), 'high': _n('high'), 'medium': _n('medium'), 'low': _n('low')}
        return {'imageName': image_name, 'isMock': False, 'summary': summ,
                'recommendation': f'Switch base to {base}-alpine or a slim tag to shrink the attack surface.',
                'fixAction': {'targetImage': f'{base}-alpine'}}
    presets = {
        'node': (3, 14, 28, 12, 'node:20-alpine'),
        'postgres': (1, 5, 12, 8, 'postgres:16-alpine'),
        'python': (2, 8, 15, 10, 'python:3.11-slim'),
    }
    c, h, m, l, target = next((v for ksub, v in presets.items() if ksub in base), (1, 3, 7, 5, f'{base}-alpine'))
    return {'imageName': image_name, 'isMock': True,
            'summary': {'critical': c, 'high': h, 'medium': m, 'low': l},
            'recommendation': f'Upgrade base image to {target} — resolves the Critical/High CVEs by moving to a minimal footprint.',
            'fixAction': {'targetImage': target}}

def docker_apply_fix(cid, target_image):
    ins, err, ok = run_cmd(['docker', 'inspect', cid])
    if not ok:
        return f'Inspect failed: {err}'
    try:
        data = json.loads(ins)[0]
        name = data['Name'].lstrip('/')
        envs = (data.get('Config') or {}).get('Env') or []
        pb = (data.get('HostConfig') or {}).get('PortBindings') or {}
        port_args = []
        for cp, binds in pb.items():
            if binds:
                port_args += ['-p', f"{binds[0]['HostPort']}:{cp.split('/')[0]}"]
        env_args = []
        for e in envs:
            env_args += ['-e', e]
        _, perr, pok = run_cmd(['docker', 'pull', target_image])
        if not pok:
            return f'Pull failed: {perr}'
        run_cmd(['docker', 'stop', cid]); run_cmd(['docker', 'rm', cid])
        cmd = ['docker', 'run', '-d', '--name', name] + port_args + env_args + [target_image]
        out, rerr, rok = run_cmd(cmd)
        return f'Upgraded -> {out.strip()[:12]}' if rok else f'Relaunch failed: {rerr}'
    except Exception as e:
        return f'Upgrade error: {e}'

print('Docker console ready.')

## 7 · Kubernetes console

Wraps `kubectl`. Lists nodes/deployments/services/pods and performs rollout-restart, scale,
delete-pod, and log fetch.

In [ ]:
def k8s_status():
    out, _, ok = run_cmd(['kubectl', 'version', '--client'])
    _, _, running = run_cmd(['kubectl', 'get', 'nodes'])
    return {'installed': ok, 'version': out.strip() or 'Not found', 'running': running,
            'context': 'connected' if running else 'Unavailable'}

def k8s_resources():
    out, err, ok = run_cmd(['kubectl', 'get', 'pods,svc,deploy,nodes', '-o', 'json', '--all-namespaces'])
    res = {'pods': [], 'services': [], 'deployments': [], 'nodes': []}
    if not ok:
        return res
    try:
        items = json.loads(out).get('items', [])
    except json.JSONDecodeError:
        return res
    for it in items:
        kind = it.get('kind'); md_ = it.get('metadata', {}); st = it.get('status', {}); sp = it.get('spec', {})
        name = md_.get('name'); ns = md_.get('namespace', 'default')
        if kind == 'Pod':
            cs = st.get('containerStatuses', []) or []
            ready = sum(1 for c in cs if c.get('ready'))
            res['pods'].append({'name': name, 'namespace': ns, 'status': st.get('phase', 'Unknown'),
                                'ready': f'{ready}/{len(cs)}', 'ip': st.get('podIP', 'None'),
                                'node': sp.get('nodeName', 'None'),
                                'restarts': sum(c.get('restartCount', 0) for c in cs)})
        elif kind == 'Service':
            ports = ', '.join(f"{p.get('port')}:{p.get('targetPort')}/{p.get('protocol')}" for p in sp.get('ports', []) or [])
            res['services'].append({'name': name, 'namespace': ns, 'type': sp.get('type', 'ClusterIP'),
                                    'clusterIp': sp.get('clusterIP', 'None'), 'ports': ports})
        elif kind == 'Deployment':
            res['deployments'].append({'name': name, 'namespace': ns,
                                       'ready': f"{st.get('readyReplicas', 0)}/{sp.get('replicas', 0)}",
                                       'replicas': sp.get('replicas', 0)})
        elif kind == 'Node':
            conds = st.get('conditions', []) or []
            ready = next((c for c in conds if c.get('type') == 'Ready'), None)
            status = ('Ready' if ready and ready.get('status') == 'True' else 'NotReady') if ready else 'Unknown'
            ip = next((a['address'] for a in st.get('addresses', []) or [] if a.get('type') == 'InternalIP'), 'Unknown')
            cap = st.get('capacity', {}) or {}
            res['nodes'].append({'name': name, 'status': status,
                                 'version': (st.get('nodeInfo') or {}).get('kubeletVersion', 'Unknown'), 'ip': ip,
                                 'gpus': cap.get('nvidia.com/gpu', '0'), 'cpu': cap.get('cpu', '?'),
                                 'mem': cap.get('memory', '?')})
    return res

def k8s_action(action, name, namespace='default', replicas=None):
    if not NAME_RE.match(name or '') or not NAME_RE.match(namespace or ''):
        return 'Invalid name/namespace'
    if action == 'restart_deploy':
        cmd = ['kubectl', 'rollout', 'restart', f'deployment/{name}', '-n', namespace]
    elif action == 'scale_deploy':
        cmd = ['kubectl', 'scale', f'deployment/{name}', f'--replicas={int(replicas)}', '-n', namespace]
    elif action == 'delete_pod':
        cmd = ['kubectl', 'delete', f'pod/{name}', '-n', namespace]
    else:
        return 'Invalid action'
    out, err, ok = run_cmd(cmd)
    return out.strip() if ok else f'Failed: {err}'

def k8s_logs(namespace, pod, tail=150):
    if not NAME_RE.match(namespace or '') or not NAME_RE.match(pod or ''):
        return 'Invalid namespace/pod'
    out, err, ok = run_cmd(['kubectl', 'logs', '-n', namespace, pod, '--tail', str(tail)])
    return out or err

print('Kubernetes console ready.')

## 8 · DevOps Agent — cluster-aware chat with approved actions

Injects live Docker/K8s state into the prompt, streams a reply, and parses `[ACTION: {...}]`
blocks the model appends. You approve each action before it runs.

In [ ]:
def gather_cluster_state():
    dv, _, _ = run_cmd(['docker', '--version'])
    kv, _, _ = run_cmd(['kubectl', 'version', '--client'])
    conts = docker_containers()
    if conts:
        dstr = 'Docker is running with the following containers:\n' + '\n'.join(
            f"- Container: Name='{c['name']}', ID='{c['id']}', Image='{c['image']}', "
            f"Status='{c['status']}', State='{c['state']}', Ports='{c['ports']}'" for c in conts)
    else:
        dstr = 'Docker status: not running or no containers.'
    r = k8s_resources()
    if any(r.values()):
        kstr = ('Kubernetes is active.\nNodes:\n' +
                '\n'.join(f"  - Node: Name='{n['name']}', Status='{n['status']}', K8sVersion='{n['version']}'" for n in r['nodes']) +
                '\nDeployments:\n' + '\n'.join(f"  - Deployment: Name='{d['name']}', Namespace='{d['namespace']}', Replicas='{d['ready']}'" for d in r['deployments']) +
                '\nServices:\n' + '\n'.join(f"  - Service: Name='{s['name']}', Namespace='{s['namespace']}', Type='{s['type']}', IP='{s['clusterIp']}', Ports='{s['ports']}'" for s in r['services']) +
                '\nPods:\n' + '\n'.join(f"  - Pod: Name='{p['name']}', Namespace='{p['namespace']}', Status='{p['status']}', Ready='{p['ready']}'" for p in r['pods']))
    else:
        kstr = 'Kubernetes status: not running or no resources.'
    return dv, kv, dstr, kstr

def _agent_system(dv, kv, dstr, kstr):
    return ('You are Kalam, a DevOps AI Agent running locally with read/write access to Docker '
            'and Kubernetes. Current live cluster state:\n---\n'
            f'- Docker Version: {dv.strip() or "Unknown"}\n- Kubernetes Client: {kv.strip() or "Unknown"}\n\n'
            f'{dstr}\n\n{kstr}\n---\n\nINSTRUCTIONS:\n'
            '1. Explain the state clearly when asked.\n'
            '2. For relationships/topology, generate a Mermaid diagram in a ```mermaid code block '
            '(subgraphs for namespaces / Docker vs K8s, arrows for port/hosting relationships).\n'
            '3. To recommend an action, append a JSON block at the VERY END using this syntax:\n'
            '   [ACTION: {"type": "docker_restart", "id": "NAME", "label": "Restart container NAME"}]\n'
            '   [ACTION: {"type": "docker_stop", "id": "NAME", "label": "Stop container NAME"}]\n'
            '   [ACTION: {"type": "docker_start", "id": "NAME", "label": "Start container NAME"}]\n'
            '   [ACTION: {"type": "k8s_restart_deploy", "name": "D", "namespace": "NS", "label": "Restart deployment D"}]\n'
            '   [ACTION: {"type": "k8s_scale", "name": "D", "namespace": "NS", "replicas": N, "label": "Scale D to N"}]\n'
            '   [ACTION: {"type": "k8s_delete_pod", "name": "P", "namespace": "NS", "label": "Delete pod P"}]\n'
            '   Only output actions that make direct sense. No placeholders.\n'
            '4. Keep answers friendly, technical, and crisp.')

def agent_chat(prompt, history=None):
    dv, kv, dstr, kstr = gather_cluster_state()
    system = _agent_system(dv, kv, dstr, kstr)
    if not llm_available():
        return ('No AI engine configured. Set a Gemini key or switch PROVIDER to local/custom. '
                'Meanwhile the Docker/Kubernetes tabs work directly.'), []
    text, ok, reason = llm_chat(system, prompt, history or [], ai_name='Kalam')
    if not ok:
        return f'⚠️ {reason}', []
    actions = []
    for m in re.finditer(r'\[ACTION:\s*(\{.*?\})\s*\]', text, re.S):
        try:
            actions.append(json.loads(m.group(1)))
        except json.JSONDecodeError:
            pass
    clean = re.sub(r'\[ACTION:\s*\{.*?\}\s*\]', '', text, flags=re.S).strip()
    return clean, actions

def run_agent_action(a):
    t = a.get('type')
    if t == 'docker_restart': return docker_action('restart', a.get('id'))
    if t == 'docker_stop':    return docker_action('stop', a.get('id'))
    if t == 'docker_start':   return docker_action('start', a.get('id'))
    if t == 'k8s_restart_deploy': return k8s_action('restart_deploy', a.get('name'), a.get('namespace', 'default'))
    if t == 'k8s_scale':      return k8s_action('scale_deploy', a.get('name'), a.get('namespace', 'default'), a.get('replicas'))
    if t == 'k8s_delete_pod': return k8s_action('delete_pod', a.get('name'), a.get('namespace', 'default'))
    return f'Unknown action type: {t}'

print('DevOps agent ready.')

## 9 · 🖥️ VM monitoring & SSH (manual inventory)

Monitors the VMs listed in `VM_INVENTORY`. For each VM it does a fast TCP reachability check,
then (over SSH) pulls **load / CPU count / memory / disk / uptime**. You can run ad-hoc
commands and open a **native SSH terminal** with one click.

SSH backend: uses **paramiko** if installed (in-notebook metrics + command exec); otherwise it
falls back to your system `ssh` binary. The *Open terminal* button always launches your OS
terminal with the right `ssh` command.

In [ ]:
import socket, shlex, platform, os

try:
    import paramiko
    HAVE_PARAMIKO = True
except ImportError:
    HAVE_PARAMIKO = False

def vm_reachable(vm, timeout=3):
    try:
        with socket.create_connection((vm['host'], vm.get('port', 22)), timeout=timeout):
            return True
    except OSError:
        return False

def _ssh_cmd_args(vm):
    args = ['ssh', '-o', 'BatchMode=yes', '-o', 'StrictHostKeyChecking=accept-new',
            '-o', 'ConnectTimeout=8', '-p', str(vm.get('port', 22))]
    if vm.get('key'):
        args += ['-i', os.path.expanduser(vm['key'])]
    args += [f"{vm['user']}@{vm['host']}"]
    return args

def ssh_run(vm, command, timeout=20):
    '''Run a remote command. Returns (stdout, stderr, ok).'''
    if HAVE_PARAMIKO:
        cli = paramiko.SSHClient()
        cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        try:
            cli.connect(vm['host'], port=vm.get('port', 22), username=vm['user'],
                        key_filename=os.path.expanduser(vm['key']) if vm.get('key') else None,
                        password=vm.get('password'), timeout=8, allow_agent=True, look_for_keys=True)
            _, out, err = cli.exec_command(command, timeout=timeout)
            o, e = out.read().decode(errors='replace'), err.read().decode(errors='replace')
            cli.close()
            return o, e, True
        except Exception as ex:
            return '', str(ex), False
    # Fallback: system ssh
    return run_cmd(_ssh_cmd_args(vm) + [command])

_METRIC_CMD = (
    "echo HOST:$(hostname); "
    "echo LOAD:$(cat /proc/loadavg 2>/dev/null | awk '{print $1}'); "
    "echo NCPU:$(nproc 2>/dev/null); "
    "echo MEM:$(free -m 2>/dev/null | awk 'NR==2{print $3\"/\"$2\" MB\"}'); "
    "echo DISK:$(df -h / 2>/dev/null | awk 'NR==2{print $5\" used (\"$3\"/\"$2\")\"}'); "
    "echo UP:$(uptime -p 2>/dev/null)"
)

def vm_metrics(vm):
    m = {'name': vm['name'], 'host': vm['host'], 'reachable': vm_reachable(vm)}
    if not m['reachable']:
        m['error'] = 'port unreachable'
        return m
    out, err, ok = ssh_run(vm, _METRIC_CMD)
    if not ok:
        m['error'] = err.strip()[:200] or 'ssh failed'
        return m
    for line in out.splitlines():
        if ':' in line:
            k, _, v = line.partition(':')
            m[k.strip().lower()] = v.strip()
    return m

def open_ssh_terminal(vm):
    '''Launch a native OS terminal with an interactive ssh session.'''
    inner = ' '.join(shlex.quote(a) for a in _ssh_cmd_args(vm)[:-1] + [f"{vm['user']}@{vm['host']}"])
    # drop BatchMode for interactive (allow password/agent prompts)
    inner = inner.replace('-o BatchMode=yes ', '')
    sysname = platform.system()
    try:
        if sysname == 'Windows':
            subprocess.Popen(['cmd', '/c', 'start', 'cmd', '/k', inner], shell=True)
        elif sysname == 'Darwin':
            subprocess.Popen(['osascript', '-e', f'tell app "Terminal" to do script "{inner}"'])
        else:
            for term in (['x-terminal-emulator', '-e'], ['gnome-terminal', '--'], ['xterm', '-e']):
                try:
                    subprocess.Popen(term + inner.split()); break
                except FileNotFoundError:
                    continue
        return f'Opened SSH terminal: {inner}'
    except Exception as e:
        return f'Could not open terminal ({e}). Run manually:\n  {inner}'

print(f'VM monitor ready (paramiko={"yes" if HAVE_PARAMIKO else "no, using system ssh"}). VMs: {len(VM_INVENTORY)}')

## 9.5 · 🗺️ PCAI Stack Visualizer

The heart of the console: **see the whole HPE Private Cloud AI deployment at once.** It reads
your live Kubernetes cluster, classifies every workload into the PCAI logical layer it belongs
to (**MLDE / MLDM / MLIS / Data Lakehouse / Identity / GPU Operator / Ingress / Platform**),
maps those onto the GPU nodes, and renders one Mermaid topology tying together the GreenLake
control plane → Kubernetes platform → AI Essentials services → the served **model endpoint** →
your managed **VMs**.

Classification is heuristic (by resource name/namespace), so it works on any PCAI cluster
without extra config. Refine `PCAI_COMPONENTS` below if your names differ.

In [ ]:
import uuid, html as _html
from IPython.display import HTML, display

# name/namespace substrings -> PCAI logical component (order matters; first match wins)
PCAI_COMPONENTS = [
    ('MLIS · Inference',      ['mlis', 'aioli', 'kserve', 'inference', 'nim', 'knative', 'serving']),
    ('MLDM · Data Mgmt',      ['mldm', 'pachyderm', 'pachd']),
    ('MLDE · Training',       ['mlde', 'determined']),
    ('Data Lakehouse',        ['ezpresto', 'presto', 'trino', 'lakehouse', 'spark', 'airflow', 'superset', 'mlflow', 'feast']),
    ('Identity · Keycloak',   ['keycloak', 'oidc', 'dex', 'auth']),
    ('GPU Operator',          ['nvidia', 'gpu-operator', 'device-plugin', 'dcgm']),
    ('Ingress / Network',     ['ingress', 'istio', 'nginx', 'metallb', 'cert-manager', 'gateway']),
]

def _classify(name, ns):
    hay = f'{ns} {name}'.lower()
    for label, subs in PCAI_COMPONENTS:
        if any(s in hay for s in subs):
            return label
    return 'Platform / Other'

def pcai_inventory():
    '''Roll up live cluster state into PCAI logical components.'''
    r = k8s_resources()
    comps = {}
    for kind in ('deployments', 'pods', 'services'):
        for item in r.get(kind, []):
            c = _classify(item['name'], item.get('namespace', 'default'))
            comps.setdefault(c, {'deployments': [], 'pods': [], 'services': []})[kind].append(item)
    total_gpu = 0
    for n in r['nodes']:
        try:
            total_gpu += int(str(n.get('gpus', '0')))
        except ValueError:
            pass
    return {'nodes': r['nodes'], 'components': comps, 'total_gpu': total_gpu,
            'ready_nodes': sum(1 for n in r['nodes'] if n['status'] == 'Ready'),
            'total_pods': len(r['pods']), 'running_pods': sum(1 for p in r['pods'] if p['status'] == 'Running')}

def _sid(s):
    return 'n' + re.sub(r'[^a-zA-Z0-9]', '', str(s))[:40]

def build_pcai_mermaid(inv=None):
    inv = inv or pcai_inventory()
    L = ['flowchart TB', '  GL["🌐 HPE GreenLake<br/>Control Plane"]']
    # model endpoint node
    if PROVIDER == 'gemini':
        ep = f'Gemini · {GEMINI_MODEL}'
    elif PROVIDER == 'custom':
        ep = f'Custom Endpoint<br/>{CUSTOM_MODEL}'
    else:
        ep = f'Local · {LOCAL_MODEL}'
    L.append(f'  ME["🧠 Model Endpoint<br/>{ep}"]')
    # PCAI platform + K8s nodes
    L.append('  subgraph PCAI["🏛️ HPE Private Cloud AI"]')
    L.append('    direction TB')
    L.append('    subgraph K8S["☸️ Kubernetes Platform"]')
    if inv['nodes']:
        for n in inv['nodes']:
            gpu = f" · {n.get('gpus','0')}×GPU" if str(n.get('gpus','0')) not in ('0', '') else ''
            L.append(f'      {_sid(n["name"])}["🖧 {n["name"]}<br/>{n["status"]}{gpu}"]')
    else:
        L.append('      NONODE["(no nodes — kubectl not connected)"]')
    L.append('    end')
    # component subgraphs
    icons = {'MLIS · Inference': '🚀', 'MLDM · Data Mgmt': '🗂️', 'MLDE · Training': '🎓',
             'Data Lakehouse': '🏞️', 'Identity · Keycloak': '🔐', 'GPU Operator': '🎮',
             'Ingress / Network': '🌉', 'Platform / Other': '⚙️'}
    order = [c[0] for c in PCAI_COMPONENTS] + ['Platform / Other']
    present = [c for c in order if c in inv['components']]
    for c in present:
        data = inv['components'][c]
        nd, npd, nsv = len(data['deployments']), len(data['pods']), len(data['services'])
        L.append(f'    subgraph {_sid(c)}["{icons.get(c,"⚙️")} {c}"]')
        L.append(f'      {_sid(c)}i["{nd} deploy · {npd} pods · {nsv} svc"]')
        L.append('    end')
    L.append('  end')
    # VMs
    if VM_INVENTORY:
        L.append('  subgraph VMS["🖥️ Managed VMs"]')
        for vm in VM_INVENTORY:
            L.append(f'    {_sid(vm["name"])}vm["{vm["name"]}<br/>{vm["host"]}"]')
        L.append('  end')
    # edges
    L.append('  GL --> PCAI')
    for c in present:
        L.append(f'  K8S --> {_sid(c)}')
    if 'MLIS · Inference' in present:
        L.append(f'  {_sid("MLIS · Inference")} --> ME')
    else:
        L.append('  PCAI --> ME')
    if VM_INVENTORY:
        L.append('  GL -.-> VMS')
    # styling
    L.append('  classDef gl fill:#0b7285,stroke:#0b7285,color:#fff;')
    L.append('  classDef ep fill:#5f3dc4,stroke:#5f3dc4,color:#fff;')
    L.append('  class GL gl;')
    L.append('  class ME ep;')
    return '\n'.join(L)

def render_mermaid(code, theme='dark'):
    '''Render a Mermaid diagram inside the notebook (uses mermaid CDN).'''
    gid = 'mmd_' + uuid.uuid4().hex[:8]
    safe = _html.escape(code)
    tmpl = (
        '<div id="{gid}" style="background:#0d1117;padding:12px;border-radius:8px;overflow:auto">'
        '<pre class="mermaid" style="background:transparent">{code}</pre></div>'
        '<script type="module">'
        'import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";'
        'mermaid.initialize({{startOnLoad:false, theme:"{theme}", securityLevel:"loose"}});'
        'const el=document.querySelector("#{gid} .mermaid");'
        'try{{await mermaid.run({{nodes:[el]}});}}catch(e){{el.textContent="Mermaid render error: "+e;}}'
        '</script>')
    display(HTML(tmpl.format(gid=gid, code=safe, theme=theme)))

def pcai_summary_text(inv=None):
    inv = inv or pcai_inventory()
    lines = [f"Nodes: {inv['ready_nodes']}/{len(inv['nodes'])} Ready · GPUs: {inv['total_gpu']} · "
             f"Pods: {inv['running_pods']}/{inv['total_pods']} Running",
             'Components detected:']
    if inv['components']:
        for c, d in inv['components'].items():
            lines.append(f"  • {c}: {len(d['deployments'])} deploy, {len(d['pods'])} pods, {len(d['services'])} svc")
    else:
        lines.append('  (no workloads — is kubectl pointed at your PCAI cluster?)')
    return '\n'.join(lines)

print('PCAI Stack Visualizer ready. Try: render_mermaid(build_pcai_mermaid())')

## 10 · 🚀 The Console UI

Run this cell to launch the tabbed dashboard. Re-run cell **2 (Config)** and then this cell
if you change providers or VMs.

In [ ]:
import ipywidgets as W
from IPython.display import display, Markdown, clear_output

# ---- PCAI Stack Visualizer tab (primary view) ----
def pcai_viz_tab():
    diagram = W.Output(); summary = W.Output(); explain = W.Output()
    refresh = W.Button(description='Render PCAI stack', icon='sitemap', button_style='primary')
    explain_btn = W.Button(description='Explain this stack (AI)', icon='robot')
    src_btn = W.Button(description='Show Mermaid source', icon='code')
    def render(_=None):
        inv = pcai_inventory()
        with summary:
            clear_output(); print(pcai_summary_text(inv))
        with diagram:
            clear_output(); render_mermaid(build_pcai_mermaid(inv))
    def do_explain(_):
        with explain:
            clear_output(); print('Analyzing...')
            inv = pcai_inventory()
            prompt = ('Here is a live snapshot of my HPE Private Cloud AI cluster:\n' +
                      pcai_summary_text(inv) +
                      '\n\nGive me a short health read of the PCAI stack: what components are up, '
                      'any gaps or risks (GPU capacity, missing services), and the top 3 things to check.')
            ans, _ = pcai_chat(prompt, 'ask') if not llm_available() else agent_chat(prompt)
            clear_output(); display(Markdown(ans))
    def show_src(_):
        with explain:
            clear_output(); print(build_pcai_mermaid())
    refresh.on_click(render); explain_btn.on_click(do_explain); src_btn.on_click(show_src)
    render()
    return W.VBox([W.HBox([refresh, explain_btn, src_btn]), summary, diagram, explain])

# ---- Dashboard tab ----
def dashboard_tab():
    out = W.Output()
    btn = W.Button(description='Refresh status', icon='refresh', button_style='primary')
    def refresh(_=None):
        with out:
            clear_output()
            d, k = docker_status(), k8s_status()
            print('🐳 DOCKER')
            print(f"   installed: {d['installed']}   running: {d['running']}   {d['version']}")
            print(f"   containers: {len(docker_containers())}")
            print('\n☸️  KUBERNETES')
            print(f"   installed: {k['installed']}   running: {k['running']}   context: {k['context']}")
            r = k8s_resources()
            print(f"   nodes:{len(r['nodes'])}  deploys:{len(r['deployments'])}  "
                  f"svcs:{len(r['services'])}  pods:{len(r['pods'])}")
            print(f"\n🤖 LLM provider: {PROVIDER}   available: {llm_available()}")
            print(f"📚 PCAI KB: {len(KB['chunks'])} chunks  (provider={KB['embedProvider']})")
            print(f"🖥️  VMs configured: {len(VM_INVENTORY)}")
    btn.on_click(refresh); refresh()
    return W.VBox([btn, out])

# ---- Docker tab ----
def docker_tab():
    out = W.Output(); logs = W.Output()
    refresh = W.Button(description='Refresh', icon='refresh')
    box = W.VBox()
    def render(_=None):
        rows = []
        for c in docker_containers():
            lbl = W.HTML(f"<b>{c['name']}</b> <code>{c['id'][:12] if c['id'] else ''}</code><br>"
                         f"<small>{c['image']} · {c['status']}</small>")
            mk = lambda act, style, cid=c['id']: (lambda _b: (docker_action(act, cid), render()))
            bs = W.HBox([
                W.Button(description='Start', button_style='success', layout=W.Layout(width='70px')),
                W.Button(description='Stop', button_style='warning', layout=W.Layout(width='70px')),
                W.Button(description='Restart', layout=W.Layout(width='80px')),
                W.Button(description='Remove', button_style='danger', layout=W.Layout(width='80px')),
                W.Button(description='Logs', button_style='info', layout=W.Layout(width='70px')),
            ])
            bs.children[0].on_click(mk('start', 's')); bs.children[1].on_click(mk('stop', 's'))
            bs.children[2].on_click(mk('restart', 's')); bs.children[3].on_click(mk('remove', 's'))
            def show_logs(_b, cid=c['id']):
                with logs:
                    clear_output(); print(docker_logs(cid))
            bs.children[4].on_click(show_logs)
            rows.append(W.VBox([lbl, bs], layout=W.Layout(border='1px solid #30363d', padding='6px', margin='3px 0')))
        box.children = rows or [W.HTML('<i>No containers (or Docker not running).</i>')]
    # scan
    scan_in = W.Text(placeholder='image[:tag] to scan e.g. node:18', layout=W.Layout(width='260px'))
    scan_btn = W.Button(description='CVE Scan', icon='shield', button_style='danger')
    scan_out = W.Output()
    def do_scan(_):
        with scan_out:
            clear_output()
            r = docker_scan(scan_in.value.strip() or 'node:18')
            s = r['summary']
            print(f"{r['imageName']}  {'(heuristic)' if r['isMock'] else '(docker scout)'}")
            print(f"  Critical {s['critical']}  High {s['high']}  Medium {s['medium']}  Low {s['low']}")
            print(f"\n💡 {r['recommendation']}")
            print(f"   suggested target image: {r['fixAction']['targetImage']}")
    scan_btn.on_click(do_scan)
    refresh.on_click(render); render()
    return W.VBox([refresh, box, W.HTML('<hr><b>Vulnerability scan</b>'),
                   W.HBox([scan_in, scan_btn]), scan_out, W.HTML('<b>Logs</b>'), logs])

# ---- Kubernetes tab ----
def k8s_tab():
    out = W.Output()
    btn = W.Button(description='Refresh resources', icon='refresh', button_style='primary')
    def refresh(_=None):
        with out:
            clear_output()
            r = k8s_resources()
            if not any(r.values()):
                print('No resources (or kubectl not connected).'); return
            print('NODES'); [print(f"  {n['name']:24} {n['status']:9} {n['version']}  {n['ip']}") for n in r['nodes']]
            print('\nDEPLOYMENTS'); [print(f"  {d['namespace']}/{d['name']:24} ready {d['ready']}") for d in r['deployments']]
            print('\nSERVICES'); [print(f"  {s['namespace']}/{s['name']:24} {s['type']:12} {s['clusterIp']:16} {s['ports']}") for s in r['services']]
            print('\nPODS'); [print(f"  {p['namespace']}/{p['name']:34} {p['status']:10} ready {p['ready']} restarts {p['restarts']}") for p in r['pods']]
    btn.on_click(refresh); refresh()
    # actions
    ns = W.Text(value='default', description='ns', layout=W.Layout(width='180px'))
    nm = W.Text(description='name', layout=W.Layout(width='240px'))
    rep = W.IntText(value=1, description='replicas', layout=W.Layout(width='150px'))
    act_out = W.Output()
    b_rs = W.Button(description='Rollout restart'); b_sc = W.Button(description='Scale'); b_dp = W.Button(description='Delete pod', button_style='danger')
    b_lg = W.Button(description='Pod logs', button_style='info')
    def _do(fn):
        def h(_):
            with act_out: clear_output(); print(fn())
            refresh()
        return h
    b_rs.on_click(_do(lambda: k8s_action('restart_deploy', nm.value, ns.value)))
    b_sc.on_click(_do(lambda: k8s_action('scale_deploy', nm.value, ns.value, rep.value)))
    b_dp.on_click(_do(lambda: k8s_action('delete_pod', nm.value, ns.value)))
    b_lg.on_click(lambda _: (act_out.clear_output(), act_out.append_stdout(k8s_logs(ns.value, nm.value))))
    return W.VBox([btn, out, W.HTML('<hr><b>Actions</b>'),
                   W.HBox([ns, nm, rep]), W.HBox([b_rs, b_sc, b_dp, b_lg]), act_out])

# ---- PCAI tab ----
def pcai_tab():
    mode = W.ToggleButtons(options=[('Ask', 'ask'), ('Diagnose Error', 'diagnose')], value='ask')
    q = W.Textarea(placeholder='Ask about HPE PCAI, or paste an error/log to diagnose...',
                   layout=W.Layout(width='99%', height='90px'))
    send = W.Button(description='Ask PCAI', icon='paper-plane', button_style='primary')
    train_btn = W.Button(description='Train / Refresh KB', icon='refresh')
    out = W.Output()
    def do_send(_):
        with out:
            clear_output(); print('Thinking...')
            ans, srcs = pcai_chat(q.value.strip(), mode.value)
            clear_output()
            display(Markdown(ans))
            if srcs:
                display(Markdown('\n**Sources**\n' + '\n'.join(
                    f"- [[{s['ref']}]] [{s['title']}]({s['url']}) · score {s['score']}" for s in srcs)))
    def do_train(_):
        with out:
            clear_output(); train(progress=print)
    send.on_click(do_send); train_btn.on_click(do_train)
    return W.VBox([W.HBox([mode, train_btn]), q, send, out])

# ---- DevOps Agent tab ----
def agent_tab():
    hist = []
    q = W.Textarea(placeholder='e.g. "draw my cluster topology" or "restart the nginx container"',
                   layout=W.Layout(width='99%', height='70px'))
    send = W.Button(description='Send', icon='paper-plane', button_style='primary')
    out = W.Output(); act_box = W.VBox()
    def do_send(_):
        with out:
            clear_output(); print('Thinking...')
            ans, actions = agent_chat(q.value.strip(), hist)
            hist.append({'role': 'user', 'content': q.value.strip()})
            hist.append({'role': 'assistant', 'content': ans})
            clear_output(); display(Markdown(ans))
        rows = []
        for a in actions:
            b = W.Button(description='✔ ' + a.get('label', a.get('type', 'action')), button_style='warning')
            def mk(a2):
                def h(_):
                    with out: print('\n▶', run_agent_action(a2))
                return h
            b.on_click(mk(a)); rows.append(b)
        act_box.children = rows
    send.on_click(do_send)
    return W.VBox([q, send, W.HTML('<b>Suggested actions (click to approve & run)</b>'), act_box, out])

# ---- VM tab ----
def vm_tab():
    box = W.VBox(); out = W.Output()
    refresh = W.Button(description='Scan VMs', icon='refresh', button_style='primary')
    cmd_in = W.Text(placeholder='remote command to run on selected VM', layout=W.Layout(width='60%'))
    def render(_=None):
        if not VM_INVENTORY:
            box.children = [W.HTML('<i>No VMs configured. Add entries to <code>VM_INVENTORY</code> '
                                   'in the Config cell and re-run it, then re-run the UI cell.</i>')]
            return
        rows = []
        for vm in VM_INVENTORY:
            m = vm_metrics(vm)
            ok = m.get('reachable')
            dot = '🟢' if ok and 'error' not in m else ('🟠' if ok else '🔴')
            info = (f"{dot} <b>{vm['name']}</b> <code>{vm['user']}@{vm['host']}:{vm.get('port',22)}</code>")
            if 'error' in m:
                detail = f"<small style='color:#e06c75'>{m['error']}</small>"
            else:
                detail = (f"<small>load <b>{m.get('load','?')}</b> · cpu <b>{m.get('ncpu','?')}</b> · "
                          f"mem <b>{m.get('mem','?')}</b> · disk <b>{m.get('disk','?')}</b> · "
                          f"up {m.get('up','?')}</small>")
            term = W.Button(description='Open SSH', icon='terminal', button_style='success', layout=W.Layout(width='120px'))
            runb = W.Button(description='Run cmd', icon='play', layout=W.Layout(width='110px'))
            def mk_term(v):
                return lambda _: out.append_stdout('\n' + open_ssh_terminal(v) + '\n')
            def mk_run(v):
                def h(_):
                    c = cmd_in.value.strip() or 'uptime'
                    o, e, okk = ssh_run(v, c)
                    with out:
                        print(f"\n$ [{v['name']}] {c}\n{o}{('[stderr] '+e) if e else ''}")
                return h
            term.on_click(mk_term(vm)); runb.on_click(mk_run(vm))
            rows.append(W.VBox([W.HTML(info + '<br>' + detail), W.HBox([term, runb])],
                               layout=W.Layout(border='1px solid #30363d', padding='6px', margin='3px 0')))
        box.children = rows
    refresh.on_click(render); render()
    return W.VBox([W.HBox([refresh, cmd_in]), box, W.HTML('<b>Output</b>'), out])

# ---- assemble ----
tabs = W.Tab()
tabs.children = [pcai_viz_tab(), dashboard_tab(), docker_tab(), k8s_tab(), pcai_tab(), agent_tab(), vm_tab()]
for i, t in enumerate(['🗺️ PCAI Stack', 'Dashboard', 'Docker', 'Kubernetes', 'PCAI Assistant', 'DevOps Agent', '🖥️ VMs']):
    tabs.set_title(i, t)
display(W.HTML('<h2>🛰️ Kalam — HPE Private Cloud AI Visualizer & Console</h2>'))
display(tabs)

## Appendix · Programmatic use (no UI)

```python
# PCAI Q&A / diagnosis
print(pcai_chat("How do I connect an external S3 bucket to the lakehouse?", "ask")[0])
print(pcai_chat("pod stuck in ImagePullBackOff on an air-gapped cluster", "diagnose")[0])

# DevOps agent
answer, actions = agent_chat("show me my cluster topology as a mermaid diagram")
print(answer)

# Docker / K8s
docker_containers(); docker_scan("python:3.11")
k8s_resources(); k8s_action("scale_deploy", "web", "default", 3)

# VMs
[vm_metrics(v) for v in VM_INVENTORY]
ssh_run(VM_INVENTORY[0], "nvidia-smi")
```

Switch engines by editing the **Config** cell (`PROVIDER = 'gemini' | 'local' | 'custom'`)
and re-running it — no other change needed.